# STARE

Source to Target Automatic Rotating Estimation (STARE) is a publicly available, blood-free quantification approach for PET tracers with irreversible kinetics. See the [preprint](https://biorxiv.org/content/10.1101/2021.09.15.460504v1.full) or [publication](https://doi.org/10.1016/j.neuroimage.2022.118901) for more details. This repository is the python implementation of it. This notebook is an example of its use.


In [ ]:
""" Preliminaries, for developing and debugging in a python notebook.
    None of this is strictly necessary, but is useful for development.
"""

# Set up the path so our code is accessible w/o installing from pypi.
import sys, os
if os.path.abspath("../src") not in sys.path:
    sys.path.insert(0, os.path.abspath("../src"))

# Force automatic re-load of libraries on every call
# (useful for accelerated debugging of oft-changed library code)
%load_ext autoreload
%autoreload 2

print(f"Executing notebook from '{os.getcwd()}'")
print("System paths available:")
for p in sys.path:
    print(f"  {p}")

In [ ]:
""" Inputs

    Specify the arguments we would be submitting to the command-line
    version of 'stare' if we were using it.
"""

from starelib.stare_wrapper import get_argument_parser, validate_arguments


# Input data for running in this notebook
local_arguments = [
    "NHFDG002",
    "--input-path", "/home/mike/projects/stare/stare_data/sampleData",
    "--output-path", "/home/mike/projects/stare/stare_data/test_python",
    "--axial-slices-to-clip", "6",
]

parser = get_argument_parser()
args = parser.parse_args(local_arguments)
if validate_arguments(args):
    print("Arguments are valid, OK to continue")
else:
    print("Fix the arguments before proceeding.")
print("Accepted arguments (after validation, including unspecified defaults)")
for k in vars(args):
    print(f"  {k}: {vars(args)[k]}")


In this cell, we simply replicate the beginning of the 'stare' 
function in 'starelib/stare_wrapper.py'

We could simply call it, but that would be the same as running the whole
multi-step process from the command line.
This notebook allows us to run one step
at a time and preview outputs in a debugger before continuing.

In [ ]:
import logging
from datetime import datetime
from starelib.stare_results import StareResults
from starelib.loader import gather_data


# Validate out_path argument
begin_timestamp = datetime.now()
logging.info(f"Begin STARE.")

# Create a StareResults object to contain and organize all of the data
results = StareResults(
    "STARE", f"STARE Results for {args.subject}", args
)

# Read data
results = gather_data(results)

# Now that we've gathered data, we should be able to pull them out of the
# results object.
print("TACs")
print(results.tacs)
print("Mid-times")
print(results.mid_times)
print("Volumes")
print(len(results.volume_images), "volumes available")

The first step in STARE is k-means clustering.

We just call the two_step_clustering function here. This allows for easy
setup of a debugger to drop in and inspect variables as it runs. It also
allows for in-notebook reporting of the results.


In [ ]:
from starelib.clustering import two_step_cluster


results = two_step_cluster(results)


In [ ]:
from starelib.partial_volume import correct_partial_volumes


results = correct_partial_volumes(results)


In [ ]:
from starelib.fit_mean_tac import fit_vascular_mean_tac
from starelib.vascular_correction import tac_vascular_correction


results = fit_vascular_mean_tac(results)
results = tac_vascular_correction(results)


In [ ]:
from starelib.boot_anchor import boot_anchor


# Change verbosity so we don't overwhelm the buffer
results.args.verbose = 0

# Create 500 artificial curves around the fitted curve we calculated earlier.
results = boot_anchor(results)


In [ ]:
# This is the most compute-intensive portion of STARE, and it
# implements the below equations to determine final parameters to the
# model.
from starelib.minimize_cost import minimize_parameter_cost


results = minimize_parameter_cost(results)


## Equations

### 1

The standard 2 tissue irreversible compartment model (2TCirr) (manuscript eq 1)

$$C_T(t)=K_1({IRF}\otimes{C_p})=K_1[(\frac{k_2}{k_2+k_3}e^{-(k_2+k_3)t} + \frac{k_3}{k_2+k_3})\otimes{C_p}](t)$$

where $t$ is time

where ${IRF}$ is the impulse response function for the target region

where $k_1$, $k_2$, $k_3$ are the microparameters of the 2TCirr model for the target region


### 2

Using equation 1 for both "source" and "target" compartments.

$$C_T(t)=K_{1,T}({IRF}_T\otimes{C_p})=K_{1,T}[(\frac{k_{2,T}}{k_{2,T}+k_{3,T}}e^{-(k_{2,T}+k_{3,T})t} + \frac{k_{3,T}}{k_{2,T}+k_{3,T}})\otimes{C_p}](t)$$

$$C_S(t)=K_{1,S}({IRF}_S\otimes{C_p})=K_{1,S}[(\frac{k_{2,S}}{k_{2,S}+k_{3,S}}e^{-(k_{2,S}+k_{3,S})t} + \frac{k_{3,S}}{k_{2,S}+k_{3,S}})\otimes{C_p}](t)$$


### 3

Define the source-to-target tissue model using a LaPlace transform to express each target compartment's TAC

$$f_T(t,\theta_{T,S})=\frac{K_{1,T}}{K_{1,S}}C_s(t) + \frac{K_{1,T}}{K_{1,S}}C_s(t)\otimes(L_{T,S}e^{{v_{T,S}}^t}+M_{T,S}e^{{\epsilon_{T,S}}^t})$$

where t is time, $\otimes$ denotes convolution, and $\theta_{T,S}$ comprises the macro-parameters $L_{T,S}$, $M_{T,S}$, $v_{T,S}$, and $\epsilon_{T,S}$.


Just checking: $\sigma\epsilon\omega\pi$ $\Sigma\Omega\Pi$

### 4

Sum the square of the residuals to determine goodness of fit.

$$\phi(t_m,\theta_{T,S})=\sum_{T=1}^{n}\sum_{m=1}^{n}w_m(C_T(t_m)-f_T(t_m,\theta_{T,S}))^2$$

where $w$ represents a set of known weights for PET frame durations.


### 5

To equation 4, add a term anchoring the estimate to PET-derived measures of vascular activity.

$$\phi(t_m,\theta_{T,S})=\sum_{T=1}^{n}\sum_{m=1}^{n}w_m(C_T(t_m)-f_T(t_m,\theta_{T,S}))^2 + \lambda\sum_{T=1}^N\lvert{K_{i,T}-K_{i,vasc,T}}\rvert$$

This is the final cost function used to fit the model.


### Altogether

From the manuscript, Figure 1

![Manuscript Figure 1](bartlett_2021_stare_figure1.jpg "Figure 1")


In [ ]:
# Wrap everything up and save the html report from this piecewise run.

results.end()
results.write_report()
results.save()

# Where did it save?
print(results.report.path)
